# FinPulse AI — Data Collection & Preparation

This notebook handles **Step 2** of the FinPulse AI pipeline:
1. Download and prepare the **Financial PhraseBank** dataset (for sentiment model)
2. Collect and label **paper abstracts** from arXiv (for recommender model)
3. Run **Great Expectations** validation on both datasets

## Why Data Quality Matters in Financial Risk
In banking, model risk management (Basel III/IV) requires that all model inputs be validated.
Garbage in = garbage out. Our Great Expectations layer enforces this principle programmatically.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import requests
from collections import Counter

# Create directories if needed
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/validated', exist_ok=True)

print('Setup complete.')

## 2. Financial PhraseBank — Sentiment Dataset

The [Financial PhraseBank](https://huggingface.co/datasets/financial_phrasebank) contains ~4,845 English sentences
from financial news, annotated by 5-8 financial domain experts as **positive**, **negative**, or **neutral**.

We use the `sentences_allagree` subset (sentences where all annotators agreed) for highest label quality.
This is the standard benchmark dataset for financial sentiment analysis.

In [ ]:
# Download Financial PhraseBank from HuggingFace
from datasets import load_dataset

dataset = load_dataset('financial_phrasebank', 'sentences_allagree', trust_remote_code=True)
df_sentiment = pd.DataFrame(dataset['train'])

# Map labels: 0=negative, 1=neutral, 2=positive
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
df_sentiment['label_text'] = df_sentiment['label'].map(label_map)

print(f'Total sentences: {len(df_sentiment)}')
print(f'\nLabel distribution:')
print(df_sentiment['label_text'].value_counts())
print(f'\nSample entries:')
df_sentiment.head()

In [ ]:
# Train/Validation split (80/20, stratified by label)
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_sentiment, test_size=0.2, random_state=42, stratify=df_sentiment['label']
)

print(f'Training set: {len(train_df)} sentences')
print(f'Validation set: {len(val_df)} sentences')
print(f'\nTraining label distribution:')
print(train_df['label_text'].value_counts(normalize=True).round(3))
print(f'\nValidation label distribution:')
print(val_df['label_text'].value_counts(normalize=True).round(3))

In [ ]:
# Save to processed data directory
train_df.to_csv('../data/processed/sentiment_train.csv', index=False)
val_df.to_csv('../data/processed/sentiment_val.csv', index=False)
df_sentiment.to_csv('../data/processed/financial_phrasebank.csv', index=False)

print('Sentiment data saved to data/processed/')

## 3. Paper Abstracts — Recommender Dataset

For the recommender model, we need paper abstracts labeled as **relevant** or **not relevant**
to Banking & Finance research.

**Strategy:**
- **Relevant papers**: Fetch from arXiv categories `q-fin.RM` (Risk Management), `q-fin.GN` (General Finance), `q-fin.CP` (Computational Finance)
- **Not relevant papers**: Fetch from unrelated arXiv categories like `cs.CV` (Computer Vision), `physics.bio-ph` (Biological Physics)

This gives us a clean binary classification task with naturally labeled data.

In [ ]:
import feedparser
import time

def fetch_arxiv_papers(query, max_results=200):
    """Fetch papers from arXiv API."""
    url = f'http://export.arxiv.org/api/query?search_query={query}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending'
    feed = feedparser.parse(url)
    
    papers = []
    for entry in feed.entries:
        papers.append({
            'title': entry.title.replace('\n', ' ').strip(),
            'abstract': entry.summary.replace('\n', ' ').strip(),
            'authors': ', '.join([a.name for a in entry.authors]),
            'date': entry.published[:10],
            'source': 'arXiv',
            'url': entry.link,
        })
    return pd.DataFrame(papers)

print('Fetching relevant papers (finance)...')
df_relevant = fetch_arxiv_papers('cat:q-fin.RM+OR+cat:q-fin.GN+OR+cat:q-fin.CP', max_results=200)
df_relevant['relevant'] = 1
print(f'  Got {len(df_relevant)} relevant papers')

time.sleep(3)  # Be polite to arXiv API

print('Fetching non-relevant papers (other fields)...')
df_irrelevant = fetch_arxiv_papers('cat:cs.CV+OR+cat:physics.bio-ph+OR+cat:math.CO', max_results=200)
df_irrelevant['relevant'] = 0
print(f'  Got {len(df_irrelevant)} non-relevant papers')

# Combine
df_papers = pd.concat([df_relevant, df_irrelevant], ignore_index=True)
df_papers = df_papers.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

print(f'\nTotal paper dataset: {len(df_papers)}')
print(f'Relevant: {df_papers["relevant"].sum()} | Not relevant: {(df_papers["relevant"]==0).sum()}')

In [ ]:
# Train/Validation split for recommender
train_papers, val_papers = train_test_split(
    df_papers, test_size=0.2, random_state=42, stratify=df_papers['relevant']
)

print(f'Training set: {len(train_papers)} papers')
print(f'Validation set: {len(val_papers)} papers')

# Save
train_papers.to_csv('../data/processed/papers_train.csv', index=False)
val_papers.to_csv('../data/processed/papers_val.csv', index=False)
df_papers.to_csv('../data/processed/paper_relevance.csv', index=False)

print('Paper data saved to data/processed/')

## 4. Data Validation with Great Expectations

We now validate both datasets to ensure they meet quality standards before model training.
This mirrors how banks validate data inputs for risk models under Basel III/IV requirements.

### Validation checks:
- No null values in critical fields
- No duplicate records
- Text fields meet minimum length requirements
- Label distributions are balanced enough for training
- Data types are correct

In [ ]:
# ---- Validate Sentiment Dataset ----
print('='*60)
print('VALIDATING: Financial PhraseBank (Sentiment Dataset)')
print('='*60)

checks = {}

# Check 1: No nulls in sentence column
checks['no_null_sentences'] = train_df['sentence'].notna().all()

# Check 2: No nulls in label column
checks['no_null_labels'] = train_df['label'].notna().all()

# Check 3: Labels are valid (0, 1, or 2)
checks['valid_labels'] = set(train_df['label'].unique()).issubset({0, 1, 2})

# Check 4: No empty sentences
checks['no_empty_sentences'] = (train_df['sentence'].str.len() > 0).all()

# Check 5: Minimum sentence length (at least 10 chars)
checks['min_sentence_length'] = (train_df['sentence'].str.len() >= 10).all()

# Check 6: No exact duplicates
checks['no_duplicates'] = not train_df.duplicated(subset=['sentence']).any()

# Check 7: All 3 classes present
checks['all_classes_present'] = len(train_df['label'].unique()) == 3

# Check 8: No class has less than 10% of samples
min_class_ratio = train_df['label'].value_counts(normalize=True).min()
checks['class_balance_ok'] = min_class_ratio > 0.05

for check_name, passed in checks.items():
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {status} | {check_name}')

print(f'\nOverall: {"✅ ALL PASSED" if all(checks.values()) else "❌ SOME FAILED"}')

In [ ]:
# ---- Validate Paper Dataset ----
print('='*60)
print('VALIDATING: Paper Abstracts (Recommender Dataset)')
print('='*60)

checks_papers = {}

# Check 1: No null titles
checks_papers['no_null_titles'] = train_papers['title'].notna().all()

# Check 2: No null abstracts
checks_papers['no_null_abstracts'] = train_papers['abstract'].notna().all()

# Check 3: No null authors
checks_papers['no_null_authors'] = train_papers['authors'].notna().all()

# Check 4: No null dates
checks_papers['no_null_dates'] = train_papers['date'].notna().all()

# Check 5: Abstracts have minimum length (50 chars)
checks_papers['abstract_min_length'] = (train_papers['abstract'].str.len() >= 50).all()

# Check 6: No duplicate titles
checks_papers['no_duplicate_titles'] = not train_papers.duplicated(subset=['title']).any()

# Check 7: Both classes present
checks_papers['both_classes_present'] = len(train_papers['relevant'].unique()) == 2

# Check 8: Class balance within acceptable range
min_ratio = train_papers['relevant'].value_counts(normalize=True).min()
checks_papers['class_balance_ok'] = min_ratio > 0.3

for check_name, passed in checks_papers.items():
    status = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {status} | {check_name}')

print(f'\nOverall: {"✅ ALL PASSED" if all(checks_papers.values()) else "❌ SOME FAILED"}')

In [ ]:
# Save validated data
if all(checks.values()):
    train_df.to_csv('../data/validated/sentiment_train.csv', index=False)
    val_df.to_csv('../data/validated/sentiment_val.csv', index=False)
    print('✅ Sentiment data saved to data/validated/')

if all(checks_papers.values()):
    train_papers.to_csv('../data/validated/papers_train.csv', index=False)
    val_papers.to_csv('../data/validated/papers_val.csv', index=False)
    print('✅ Paper data saved to data/validated/')

print('\n--- Step 2 Complete ---')
print('Next: Run notebook 01_sentiment_model.ipynb to train Model 1')